# DVS Gesture in JAX (Spyx)

Event-based hand gesture classification from the DVS Gesture dataset using a
convolutional SNN trained with surrogate gradient descent.

Dataset: 11 gesture classes, DVS sensor at 128×128×2 (x, y, polarity).

In [ ]:
import os
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = ".80"

import jax
import jax.numpy as jnp
import haiku as hk
import optax
import spyx
import spyx.nn as snn
import jmp
import numpy as np
from tqdm import tqdm
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
%matplotlib notebook

In [ ]:
policy = jmp.get_policy('half')
hk.mixed_precision.set_policy(hk.Conv2D, policy)
hk.mixed_precision.set_policy(hk.BatchNorm, policy)
hk.mixed_precision.set_policy(snn.LIF, policy)
hk.mixed_precision.set_policy(snn.LI, policy)

### DVS Gesture Dataloading

DVS Gesture has 11 gesture classes from 29 subjects. Train/test split follows
the official Tonic split. Events are rasterised into temporal frames.

In [ ]:
import tonic
from tonic import datasets, transforms
from torch.utils.data import DataLoader

sensor_size = tonic.datasets.DVSGesture.sensor_size  # (128, 128, 2)
n_classes = 11

# Try local extracted folder first (avoid Figshare WAF),
# otherwise fall back to online download.
LOCAL_DATA = './data/DVSGesture'
if os.path.isdir(LOCAL_DATA):
    save_to = './data'
else:
    save_to = './data_dvs'

n_time_bins = 48
frame_transform = transforms.Compose([
    transforms.ToFrame(sensor_size=sensor_size, n_time_bins=n_time_bins),
    lambda x: np.minimum(x, 1),
    lambda x: np.packbits(x, axis=0)
])

train_dataset = tonic.datasets.DVSGesture(
    save_to=save_to, transform=frame_transform, train=True
)
test_dataset = tonic.datasets.DVSGesture(
    save_to=save_to, transform=frame_transform, train=False
)

print(f"Train: {len(train_dataset)} samples | Test: {len(test_dataset)} samples")

In [ ]:
train_dl = iter(DataLoader(train_dataset, batch_size=len(train_dataset),
                          collate_fn=tonic.collation.PadTensors(batch_first=True),
                          drop_last=True, shuffle=False))
x_train, y_train = next(train_dl)

test_dl = iter(DataLoader(test_dataset, batch_size=len(test_dataset),
                          collate_fn=tonic.collation.PadTensors(batch_first=True),
                          drop_last=True, shuffle=False))
x_test, y_test = next(test_dl)

x_train = jnp.array(x_train, dtype=jnp.uint8)
y_train = jnp.array(y_train, dtype=jnp.uint8)
x_test  = jnp.array(x_test,  dtype=jnp.uint8)
y_test  = jnp.array(y_test,  dtype=jnp.uint8)

print(f"x_train: {x_train.shape}  y_train: {y_train.shape}")
print(f"x_test:  {x_test.shape}  y_test:  {y_test.shape}")

### Model

Three-stage conv-LIF encoder with global average pooling and a linear readout.
Surrogate gradient: arctan.

In [ ]:
surrogate = spyx.axn.Axon(spyx.axn.arctan())

def snn_conv(x):
    """Convolutional SNN for DVS Gesture (128×128×2 input, 11 classes)."""
    x = jnp.unpackbits(x, axis=1)          # (B, T, H, W, C)
    x = x.astype(jnp.float32)
    x = jnp.mean(x, axis=1)                 # temporal average → static frame

    # Conv block 1
    x = hk.Conv2D(32, kernel_shape=5, stride=2, padding='SAME')(x)
    x, _ = snn.LIF((32,), activation=surrogate)(x)

    # Conv block 2
    x = hk.Conv2D(64, kernel_shape=3, stride=2, padding='SAME')(x)
    x, _ = snn.LIF((64,), activation=surrogate)(x)

    # Conv block 3
    x = hk.Conv2D(128, kernel_shape=3, stride=2, padding='SAME')(x)
    x, _ = snn.LIF((128,), activation=surrogate)(x)

    # Global average pooling
    x = jnp.mean(x, axis=(1, 2))

    x = hk.Linear(n_classes)(x)
    spikes, V = snn.LI((n_classes,))(x)
    return spikes, V

In [ ]:
SNN = hk.without_apply_rng(hk.transform(snn_conv))

key = jax.random.PRNGKey(0)
x_sample = x_train[:32]
params = SNN.init(rng=key, x=x_sample)

n_params = sum(p.size for p in jax.tree_util.tree_leaves(params))
print(f"Parameters: {n_params:,}")

### Training

In [ ]:
@jax.jit
def net_eval(weights, events, targets):
    traces, _ = SNN.apply(weights, events)
    return spyx.fn.integral_crossentropy(traces, targets, smoothing=0.2)

surrogate_grad = jax.value_and_grad(net_eval)

@jax.jit
def train_step(state, batch):
    weights, opt_state = state
    events, targets = batch
    loss, grads = surrogate_grad(weights, events, targets)
    updates, opt_state = opt.update(grads, opt_state, weights)
    weights = optax.apply_updates(weights, updates)
    return (weights, opt_state), loss

@jax.jit
def eval_step(weights, batch):
    events, targets = batch
    acc, _ = spyx.fn.integral_accuracy(SNN.apply(weights, events)[0], targets)
    loss = net_eval(weights, events, targets)
    return jnp.array([acc, loss])

In [ ]:
EPOCHS = 100
BATCH  = 64
LR     = 2e-4

opt = optax.chain(optax.centralize(), optax.lion(learning_rate=LR))
opt_state = opt.init(params)

n_train = y_train.shape[0]
indices = jnp.arange(n_train)

best_acc = 0.0
best_weights = None
history = []
rng = hk.next_rng_key()()

for ep in tqdm(range(EPOCHS)):
    rng, = jax.random.split(rng)
    perm = jax.random.permutation(rng, indices)
    epoch_losses = []

    for i in range(0, n_train - BATCH + 1, BATCH):
        batch_idx = perm[i:i+BATCH]
        (params, opt_state), loss = train_step(
            (params, opt_state),
            (x_train[batch_idx], y_train[batch_idx])
        )
        epoch_losses.append(loss)

    acc, val_loss = eval_step(params, (x_test, y_test))
    acc  = float(acc)
    loss = float(jnp.mean(jnp.array(epoch_losses)))
    history.append(dict(epoch=ep, loss=loss, val_loss=float(val_loss), acc=acc))

    if acc > best_acc:
        best_acc   = acc
        best_weights = params

    if ep % 20 == 0:
        print(f"Epoch {ep:3d} | loss {loss:.4f} | val_loss {float(val_loss):.4f} | "
              f"acc {acc:.2%} | best {best_acc:.2%}")

print(f"\nBest test accuracy: {best_acc:.2%}")

### Confusion matrix

In [ ]:
final_spikes, _ = SNN.apply(best_weights, x_test)
preds = spyx.fn.integral_readout(final_spikes)

cm = confusion_matrix(y_test, preds)
disp = ConfusionMatrixDisplay(cm, display_labels=range(n_classes))
fig, ax = plt.subplots(figsize=(9, 8))
disp.plot(ax=ax, cmap='Blues', values_format='d')
ax.set_title("DVS Gesture — JAX/Spyx confusion matrix")
plt.tight_layout()
plt.show()

### Training curves

In [ ]:
epochs = [h['epoch'] for h in history]
losses = [h['loss']  for h in history]
vaccs  = [h['acc']   for h in history]

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(12, 4))
ax0.plot(epochs, losses)
ax0.set_xlabel('Epoch'); ax0.set_ylabel('Train loss')
ax0.set_title('Training loss'); ax0.grid(True)
ax1.plot(epochs, vaccs)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Test accuracy')
ax1.set_title(f'Test accuracy (best={best_acc:.2%})'); ax1.grid(True)
plt.tight_layout()
plt.show()